#   Silver to Gold


## Parâmetros do Ambiente - Organização do Ambiente

In [0]:
#PADRÃO: <catalog>.<camada>.<tabela>
catalog = "workspace"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

landing_path = f"/Volumes/{catalog}/{bronze_schema_name}/landing"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

# Funções gerais/ reutilizáveis

In [0]:
from pyspark.sql import functions as F, Row
from datetime import datetime

dq_results_gold = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results_gold.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=failed, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results_gold.append(
        Row(table_name=table_name, check_name=check_name, total_rows=total,
            failed_rows=dupes, passed=passed, checked_at=datetime.now())
    )
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

## 1. gold.dim_movies

**Origem:** `silver.tb_info_filmes`

**Objetivo:** Armazenar os metadados principais e descritivos de cada filme,ca dimensão central do Star Schema, pra qual a tabela fato e todas as bridges apontam.

**Colunas:** `sk_movie_id` (PK, surrogate key), `id_filme` (chave natural), `titulo`, `data_lancamento`, `ano_lancamento`, `duracao_minutos`, `idioma_original`, `status_filme`, `sinopse`.

**Regras aplicadas:**
- `tb_info_filmes` já chega deduplicada por `id_filme` da Silver, então aqui é só seleção de colunas + geração da surrogate key, nenhum tratamento de negócio novo.
- `sk_movie_id` gerado via `row_number()` ordenado por `id_filme`, garantindo uma chave estável e reprodutível entre execuções.
- Checagem de unicidade tanto de `sk_movie_id` (a PK) quanto de `id_filme` (a chave natural), confirmando que a transição Silver → Gold não introduziu duplicidade.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, col

tb_silver_info_filmes = spark.table(f"{silver_schema}.tb_info_filmes")

# Eu decidir criar a surrogate key via row_number(), ordenando pela chave de id_filme pra ser reprodutível entre execuções do job.
janela_sk_movie = Window.orderBy("id_filme")

gold_dim_movies = (
    tb_silver_info_filmes
    .select(
        "id_filme", "titulo", "data_lancamento", "ano_lancamento",
        "duracao_minutos", "idioma_original", "status_filme", "sinopse",
    )
    .withColumn("sk_movie_id", row_number().over(janela_sk_movie))
    .select(
        "sk_movie_id", "id_filme", "titulo", "data_lancamento", "ano_lancamento",
        "duracao_minutos", "idioma_original", "status_filme", "sinopse",
    )
)

#Checagens de qualidade
#Aqui estou apenas confirmando que está tudo nos conformes.
#No teste, sk_movie_id e id_filme precisam ser únicos já que tb_info_filmes já chega deduplicada da Silver.
dq_check_unique("dim_movies", "sk_movie_id_unico", gold_dim_movies, ["sk_movie_id"])
dq_check_unique("dim_movies", "id_filme_unico", gold_dim_movies, ["id_filme"])

gold_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_movies")
display(spark.table(f"{gold_schema}.dim_movies").limit(10))

## 2. gold.dim_genres

**Origem:** `silver.tb_generos`

**Objetivo:** Catálogo único e deduplicado de todos os gêneros cinematográficos.

**Colunas:** `sk_genre_id` (PK), `nome_genero`.

**Regras aplicadas:**
- `tb_generos` já chega validada e deduplicada da Silver (contra a lista fechada de 19 gêneros do TMDB); aqui é só extrair os valores distintos de `genero`, sem o vínculo com filme (esse vínculo vira a `bridge_movie_genre`).
- `sk_genre_id` via `row_number()` ordenado alfabeticamente por `nome_genero`, pra ser determinístico.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, col

tb_silver_generos = spark.table(f"{silver_schema}.tb_generos")

# Ordeno por nome_genero (já renomeado), não por "genero" (nome antigo).
janela_sk_genre = Window.orderBy("nome_genero")


# Eu decidi fazer o distinct() antes de gerar a surrogate key. Fiz isso pois  se a chave fosse gerada antes do distinct(), cada ocorrência do mesmo #gênero em filmes diferentes viraria uma linha separada, o que estaria errado.
gold_dim_genres = (
    tb_silver_generos
    .select("genero")
    .distinct()
    .withColumnRenamed("genero", "nome_genero")
    .withColumn("sk_genre_id", row_number().over(janela_sk_genre))
    .select("sk_genre_id", "nome_genero")
)

dq_check_unique("dim_genres", "sk_genre_id_unico", gold_dim_genres, ["sk_genre_id"])
dq_check_unique("dim_genres", "nome_genero_unico", gold_dim_genres, ["nome_genero"])

gold_dim_genres.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_genres")
display(spark.table(f"{gold_schema}.dim_genres").orderBy("nome_genero"))

## 3. gold.dim_people

**Origem:** `silver.tb_pessoas_empresas`, filtrando `tipo_entidade` em `('Ator', 'Diretor', 'Roteirista')`

**Objetivo:** Consolidar todas as pessoas físicas envolvidas na obra.

**Colunas:** `sk_person_id` (PK), `nome_pessoa`, `tipo_pessoa` (apenas 'Ator', 'Diretor' ou 'Roteirista').

**Regras aplicadas:**
- `Produtora` fica de fora dessa dimensão — vai para `dim_companies` na célula seguinte, já que não é pessoa física.
- Granularidade é `(nome_pessoa, tipo_pessoa)`, não só `nome_pessoa`: como `tipo_pessoa` é um campo único (não uma lista), uma mesma pessoa que atua como Ator num filme e Diretor em outro vira duas linhas distintas na dimensão — uma para cada papel. Isso mantém o mesmo padrão de granularidade que a própria `tb_pessoas_empresas` já usa desde a Silver.
- Checagem extra de domínio confirmando que `tipo_pessoa` só contém os três valores esperados (Produtora nunca deveria vazar pra cá).

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, col

tb_silver_pessoas_empresas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

# Só pessoa física entra aqui. A produtora fica de fora e vai pra dim_companies na célula seguinte.
tb_pessoas_fisicas = tb_silver_pessoas_empresas.filter(
    col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")
)

janela_sk_person = Window.orderBy("nome_pessoa", "tipo_pessoa")

# Aplico o distinct() sobre o par (pessoa, tipo), não só no nome. Isso porque o grão da dimensão é (pessoa, papel).
# Por exempllo, uma mesma pessoa que atua como Ator num filme e Diretor em outro vira duas linhas.

gold_dim_people = (
    tb_pessoas_fisicas
    .select("nome_pessoa_empresa", "tipo_entidade")
    .distinct()
    .withColumnRenamed("nome_pessoa_empresa", "nome_pessoa")
    .withColumnRenamed("tipo_entidade", "tipo_pessoa")
    .withColumn("sk_person_id", row_number().over(janela_sk_person))
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa")
)

dq_check_unique("dim_people", "sk_person_id_unico", gold_dim_people, ["sk_person_id"])
dq_check_unique("dim_people", "nome_pessoa_tipo_pessoa_unico", gold_dim_people, ["nome_pessoa", "tipo_pessoa"])
dq_check(
    "dim_people", "tipo_pessoa dentro do domínio esperado (Ator/Diretor/Roteirista)", gold_dim_people,
    col("tipo_pessoa").isin("Ator", "Diretor", "Roteirista")
)

gold_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_people")
display(spark.table(f"{gold_schema}.dim_people").limit(10))

## 4. gold.dim_companies

**Origem:** `silver.tb_pessoas_empresas`, filtrando `tipo_entidade = 'Produtora'`

**Objetivo:** Catálogo único de produtoras/estúdios.

**Colunas:** `sk_company_id` (PK), `nome_produtora`.

**Regras aplicadas:**
- Complementar à `dim_people`: pega exatamente o que ficou de fora de lá (`tipo_entidade = 'Produtora'`).
- `sk_company_id` via `row_number()` ordenado alfabeticamente por `nome_produtora`.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, col

tb_silver_pessoas_empresas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

# Aqui eu pego exatamente o que ficou de fora da dim_people (tipo_entidade = 'Produtora').
tb_produtoras = tb_silver_pessoas_empresas.filter(col("tipo_entidade") == "Produtora")

janela_sk_company = Window.orderBy("nome_produtora")

# Faço o distinct() antes de gerar a surrogate key, já que dim_companies é um catálogo, uma linha por produtora, não uma linha por participação em #filme.
gold_dim_companies = (
    tb_produtoras
    .select("nome_pessoa_empresa")
    .distinct()
    .withColumnRenamed("nome_pessoa_empresa", "nome_produtora")
    .withColumn("sk_company_id", row_number().over(janela_sk_company))
    .select("sk_company_id", "nome_produtora")
)

dq_check_unique("dim_companies", "sk_company_id_unico", gold_dim_companies, ["sk_company_id"])
dq_check_unique("dim_companies", "nome_produtora_unico", gold_dim_companies, ["nome_produtora"])

gold_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_companies")
display(spark.table(f"{gold_schema}.dim_companies").limit(10))

## 5. gold.bridge_movie_genre

**Origem:** `silver.tb_generos`, cruzada com `gold.dim_movies` e `gold.dim_genres`

**Objetivo:** Resolver a relação N:N entre filme e gênero, um filme pode ter vários gêneros, e um gênero pertence a vários filmes, então não cabe como coluna direta em nenhuma das duas dimensões.

**Colunas:** `sk_movie_id` (FK para `dim_movies`), `sk_genre_id` (FK para `dim_genres`).

**Regras aplicadas:**
- Join duplo: `id_filme` → `sk_movie_id` via `dim_movies`, e `genero` → `sk_genre_id` via `dim_genres` (por nome).
- `inner join` nos dois: se um `id_filme` ou um `genero` de `tb_generos` não bater com nenhuma linha da dimensão correspondente, a linha simplesmente não entra na bridge, não faz sentido ter uma FK apontando pra uma dimensão sem registro correspondente.
- `.distinct()` como rede de segurança contra combinação repetida (ex.: se por algum motivo o mesmo par filme-gênero aparecesse mais de uma vez na Silver).
- `dq_check_unique` no par `(sk_movie_id, sk_genre_id)`: garante que a bridge não tem a mesma associação duplicada.

In [0]:
from pyspark.sql.functions import col

tb_silver_generos = spark.table(f"{silver_schema}.tb_generos")
gold_dim_movies_ref = spark.table(f"{gold_schema}.dim_movies").select("id_filme", "sk_movie_id")
gold_dim_genres_ref = spark.table(f"{gold_schema}.dim_genres").select("nome_genero", "sk_genre_id")

# Aqui eu preciso fazer dois joins, um pra trocar id_filme pela sk_movie_id (usando dim_movies) e outro pra
# trocar genero pela sk_genre_id (usando dim_genres) e é assim que a bridge conecta as duas dimensões sem duplicar nada.
# Escolhi inner join nos dois de propósito. Se por algum motivo um id_filme de tb_generos não existir em dim_movies, 
# ou o texto do genero não bater com nenhum nome_genero de dim_genres, eu
# prefiro que essa linha simplesmente não entre na bridge, em vez de forçar ela a entrar com uma
# FK quebrada. Eu acredito que não faz sentido eu guardar uma referência apontando pra uma dimensão que não tem
# o registro correspondente, já que isso só ia gerar inconsistência mais na frente, quando alguém fosse
# usar a bridge pra juntar com as dimensões de novo.
gold_bridge_movie_genre = (
    tb_silver_generos
    .join(gold_dim_movies_ref, on="id_filme", how="inner")
    .join(gold_dim_genres_ref, tb_silver_generos["genero"] == gold_dim_genres_ref["nome_genero"], how="inner")
    .select("sk_movie_id", "sk_genre_id")
    .distinct()
)

dq_check_unique("bridge_movie_genre", "par_sk_movie_sk_genre_unico", gold_bridge_movie_genre, ["sk_movie_id", "sk_genre_id"])

gold_bridge_movie_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_genre")
display(spark.table(f"{gold_schema}.bridge_movie_genre").limit(10))

## 6. gold.bridge_movie_person

**Origem:** `silver.tb_pessoas_empresas`, filtrando `tipo_entidade` em `('Ator', 'Diretor', 'Roteirista')`, cruzada com `gold.dim_movies` e `gold.dim_people`

**Objetivo:** Resolver a relação N:N entre filme e pessoa (ator/diretor/roteirista), um filme tem várias pessoas envolvidas, e a mesma pessoa participa de vários filmes.

**Colunas:** `sk_movie_id` (FK para `dim_movies`), `sk_person_id` (FK para `dim_people`).

**Regras aplicadas:**
- Mesmo filtro já usado em `dim_people`: só pessoa física entra aqui, `Produtora` vai para `bridge_movie_company` na célula seguinte.
- Join com `dim_movies` por `id_filme` (igual `bridge_movie_genre`).
- Join com `dim_people` pelo **par** `(nome_pessoa, tipo_pessoa)`, não só pelo nome. Isso é obrigatório porque `dim_people` tem granularidade `(pessoa, papel)`, como documentado na criação da dimensão: casar só por nome juntaria erroneamente, numa única `sk_person_id`, as participações de uma mesma pessoa como Ator em um filme e como Diretor em outro.
- `inner join` nos dois: linha sem correspondência em alguma dimensão fica de fora.
- `.distinct()` como rede de segurança e `dq_check_unique` no par `(sk_movie_id, sk_person_id)`.

In [0]:
from pyspark.sql.functions import col

tb_silver_pessoas_empresas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

tb_pessoas_fisicas = tb_silver_pessoas_empresas.filter(
    col("tipo_entidade").isin("Ator", "Diretor", "Roteirista")
)

gold_dim_movies_ref = spark.table(f"{gold_schema}.dim_movies").select("id_filme", "sk_movie_id")
gold_dim_people_ref = spark.table(f"{gold_schema}.dim_people").select("nome_pessoa", "tipo_pessoa", "sk_person_id")

# Aqui o join com dim_people precisa ser pelo par (nome, tipo) e não só pelo nome pois a tabela
# tem uma linha por (pessoa, papel), igual explicado lá na criação da dimensão. Acredito que casar só por nome juntaria erroneamente as #participações de Ator e Diretor da mesma pessoa numa única sk_person_id.
gold_bridge_movie_person = (
    tb_pessoas_fisicas
    .join(gold_dim_movies_ref, on="id_filme", how="inner")
    .join(
        gold_dim_people_ref,
        (tb_pessoas_fisicas["nome_pessoa_empresa"] == gold_dim_people_ref["nome_pessoa"])
        & (tb_pessoas_fisicas["tipo_entidade"] == gold_dim_people_ref["tipo_pessoa"]),
        how="inner",
    )
    .select("sk_movie_id", "sk_person_id")
    .distinct()
)

dq_check_unique("bridge_movie_person", "par_sk_movie_sk_person_unico", gold_bridge_movie_person, ["sk_movie_id", "sk_person_id"])

gold_bridge_movie_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_person")
display(spark.table(f"{gold_schema}.bridge_movie_person").limit(10))

## 7. gold.bridge_movie_company

**Origem:** `silver.tb_pessoas_empresas`, filtrando `tipo_entidade = 'Produtora'`, cruzada com `gold.dim_movies` e `gold.dim_companies`

**Objetivo:** Resolver a relação N:N entre filme e produtora — um filme pode ter várias produtoras, e a mesma produtora participa de vários filmes.

**Colunas:** `sk_movie_id` (FK para `dim_movies`), `sk_company_id` (FK para `dim_companies`).

**Regras aplicadas:**
- Complementar à `bridge_movie_person`: pega exatamente o que ficou de fora de lá (`tipo_entidade = 'Produtora'`), mesmo filtro usado na criação de `dim_companies`.
- Join com `dim_movies` por `id_filme` e com `dim_companies` por `nome_produtora` (mesmo padrão das duas bridges anteriores).
- `inner join` nos dois, `.distinct()` como rede de segurança e `dq_check_unique` no par `(sk_movie_id, sk_company_id)`.


In [0]:
from pyspark.sql.functions import col

tb_silver_pessoas_empresas = spark.table(f"{silver_schema}.tb_pessoas_empresas")

# Essa bridge é o espelho da bridge_movie_person, só que pro lado das produtoras. Eu filtro aepnas
# tipo_entidade = 'Produtora', que é exatamente o que ficou de fora quando montei dim_companies.
tb_produtoras = tb_silver_pessoas_empresas.filter(col("tipo_entidade") == "Produtora")

gold_dim_movies_ref = spark.table(f"{gold_schema}.dim_movies").select("id_filme", "sk_movie_id")
gold_dim_companies_ref = spark.table(f"{gold_schema}.dim_companies").select("nome_produtora", "sk_company_id")


# Eu precisei fazer dois joins, um pra trocar id_filme pela sk_movie_id (via dim_movies) e outro pra trocar
# nome_pessoa_empresa pela sk_company_id (via dim_companies). A bridge só guarda o par de surrogate keys, nunca as chaves naturais.
# Uso depois o inner join nos dois de propósito, igual fiz nas outras bridge. Se um id_filme não existir
# em dim_movies, ou o nome da produtora não bater com nenhum nome_produtora de dim_companies, eu
# prefiro que a linha simplesmente não entre, em vez de guardar uma FK sem o registro correspondente do outro lado.
gold_bridge_movie_company = (
    tb_produtoras
    .join(gold_dim_movies_ref, on="id_filme", how="inner")
    .join(gold_dim_companies_ref, tb_produtoras["nome_pessoa_empresa"] == gold_dim_companies_ref["nome_produtora"], how="inner")
    .select("sk_movie_id", "sk_company_id")
    .distinct()
)

dq_check_unique("bridge_movie_company", "par_sk_movie_sk_company_unico", gold_bridge_movie_company, ["sk_movie_id", "sk_company_id"])

gold_bridge_movie_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.bridge_movie_company")
display(spark.table(f"{gold_schema}.bridge_movie_company").limit(10))

## 8. gold.dim_reviews

**Origem:** `silver.tb_avaliacoes_usuarios`, agregada por `id_filme`, cruzada com `gold.dim_movies`

**Objetivo:** Consolidar as avaliações individuais dos usuários numa métrica resumida por filme.

**Colunas:** `sk_review_id` (PK), `sk_movie_id` (FK para `dim_movies`), `qtd_avaliacoes_usuarios` (contagem), `nota_media_usuarios` (média arredondada em 2 casas decimais).

**Regras aplicadas:**
- `qtd_avaliacoes_usuarios` conta TODAS as avaliações recebidas pelo filme (uma linha de `tb_avaliacoes_usuarios` = uma avaliação), mesmo as que tiveram `nota_usuario` descartada (virou NULL) na Silver por estar fora da faixa 0-10, é o volume de pessoas que avaliaram, não o volume de notas numéricas válidas.
- `nota_media_usuarios` usa `avg(nota_usuario)`, que no Spark ignora NULL automaticamente em vez de tratá-lo como zero — mesmo raciocínio de propagação de NULL já usado em `tb_financeiro_filmes`, só que aqui numa agregação em vez de uma conta aritmética simples.
- `inner join` com `dim_movies`: só entram filmes que também existem na dimensão de filmes.
- `sk_review_id` via `row_number()` ordenado por `id_filme`, mesmo padrão reprodutível das demais surrogate keys da Gold.

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import col, count, avg, round as spark_round, row_number

tb_silver_avaliacoes = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")
gold_dim_movies_ref = spark.table(f"{gold_schema}.dim_movies").select("id_filme", "sk_movie_id")

# Realizo a agregação por filme onde cada avaliação de usuário vira uma métrica resumida.
# - qtd_avaliacoes_usuarios conta todas as avaliações recebidas pelo filme 
# - nota_media_usuarios usa avg(nota_usuario), que no Spark ignora NULL automaticamente em vez
#   de tratá-lo como zero, esse é o mesmo raciocínio de propagação de NULL que já usei em tb_financeiro_filmes, só que aqui numa agregação em vez de #uma conta aritmética simples.
tb_avaliacoes_agregadas = (
    tb_silver_avaliacoes
    .groupBy("id_filme")
    .agg(
        count("*").alias("qtd_avaliacoes_usuarios"),
        spark_round(avg("nota_usuario"), 2).alias("nota_media_usuarios"),
    )
)


janela_sk_review = Window.orderBy("id_filme")

gold_dim_reviews = (
    tb_avaliacoes_agregadas
    # Realizei um inner join onde só entram filmes que também existem em dim_movies.
    .join(gold_dim_movies_ref, on="id_filme", how="inner")
    .withColumn("sk_review_id", row_number().over(janela_sk_review))
    .select("sk_review_id", "sk_movie_id", "qtd_avaliacoes_usuarios", "nota_media_usuarios")
)

dq_check_unique("dim_reviews", "sk_review_id_unico", gold_dim_reviews, ["sk_review_id"])
dq_check_unique("dim_reviews", "sk_movie_id_unico", gold_dim_reviews, ["sk_movie_id"])
dq_check("dim_reviews", "qtd_avaliacoes_usuarios positivo", gold_dim_reviews, col("qtd_avaliacoes_usuarios") > 0)
dq_check(
    "dim_reviews",
    "nota_media_usuarios dentro de 0 a 10 (quando não nula)",
    gold_dim_reviews,
    (col("nota_media_usuarios").isNull()) | ((col("nota_media_usuarios") >= 0) & (col("nota_media_usuarios") <= 10)),
)

gold_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.dim_reviews")
display(spark.table(f"{gold_schema}.dim_reviews").limit(10))

## 9. gold.fact_movies_performance

**Origem:** `gold.dim_movies` como base, cruzada com `silver.tb_financeiro_filmes` e `silver.tb_metricas_engajamento`

**Grão:** Um registro único por filme (mesmo grão de `dim_movies`).

**Objetivo:** Centralizar todas as métricas financeiras e de engajamento do filme numa única tabela fato.

**Colunas:**
- `sk_movie_id` BIGINT (FK para `dim_movies`)
- Financeiras, `DECIMAL(18,2)`: `orcamento_usd`, `receita_usd`, `lucro_usd`, `orcamento_brl`, `receita_brl`, `lucro_brl`
- Engajamento — `DOUBLE`/`INT`: `popularidade` (DOUBLE), `nota_media_tmdb` (DOUBLE), `qtd_votos_tmdb` (INT), `nota_media_imdb` (DOUBLE), `qtd_votos_imdb` (INT)

**Regras aplicadas:**
- Ponto de partida é `dim_movies`, não as tabelas financeira/engajamento: `LEFT JOIN` a partir de `dim_movies` garante que todo filme da dimensão aparece na fato, mesmo os que não têm orçamento/receita preenchidos (lembrando: só ~8% dos filmes têm orçamento) ou métrica de engajamento, essas colunas simplesmente ficam NULL, em vez do filme desaparecer por causa de um inner join.
- `tb_financeiro_filmes` e `tb_metricas_engajamento` já são deduplicadas por `id_filme` desde a Silver (uma linha por filme cada), então os dois `LEFT JOIN`s não multiplicam o grão de `dim_movies` — é o "sem duplicar grão através dos joins" exigido no enunciado.
- Cast explícito pros tipos exigidos (`DECIMAL(18,2)` nas financeiras, `DOUBLE`/`INT` nas de engajamento), garantindo o contrato de tipo da tabela fato independente de como a aritmética foi resolvida na Silver.
- `margem_lucro_percentual` (existente em `tb_financeiro_filmes`) NÃO entra na fato — não está na lista de colunas pedida no enunciado para `fact_movies_performance`, então fica só na Silver.
- Checagem de grão: comparo a contagem de linhas da fato com a de `dim_movies` pra confirmar que os LEFT JOINs realmente preservaram um-pra-um, além do `dq_check_unique` em `sk_movie_id`.

In [0]:
from pyspark.sql.functions import col

gold_dim_movies_ref = spark.table(f"{gold_schema}.dim_movies").select("id_filme", "sk_movie_id")
tb_silver_financeiro = spark.table(f"{silver_schema}.tb_financeiro_filmes").select(
    "id_filme", "orcamento_usd", "receita_usd", "lucro_usd", "orcamento_brl", "receita_brl", "lucro_brl"
)
tb_silver_metricas = spark.table(f"{silver_schema}.tb_metricas_engajamento").select(
    "id_filme", "popularidade", "nota_media_tmdb", "qtd_votos_tmdb", "nota_media_imdb", "qtd_votos_imdb"
)

# Aqui o ponto de partida é dim_movies, não tb_financeiro_filmes/tb_metricas_engajamento. 
# o grão da fato é "um registro por filme que existe na dimensão". Por isso decidi usar o LEFT JOIN a partir de
# dim_movies. Temos que um filme sem dado financeiro ou sem métrica de engajamento ainda aparece na fato, só que com essas colunas em
# NULL, em vez de desaparecer da tabela por causa de um inner join.
# Fiz dessa forma mas acredito ser mais sefuro porque tb_financeiro_filmes e tb_metricas_engajamento já são deduplicadas por
# id_filme lá na Silver, então nenhum dos dois LEFT JOINs pode multiplicar o grão de dim_movies.
gold_fact_movies_performance = (
    gold_dim_movies_ref
    .join(tb_silver_financeiro, on="id_filme", how="left")
    .join(tb_silver_metricas, on="id_filme", how="left")
    .select(
        "sk_movie_id",
        # Métricas financeiras
        col("orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        col("receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        col("lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
        col("orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        col("receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        col("lucro_brl").cast("decimal(18,2)").alias("lucro_brl"),
        # Métricas de engajamento — DOUBLE/INT, conforme pedido no enunciado.
        col("popularidade").cast("double").alias("popularidade"),
        col("nota_media_tmdb").cast("double").alias("nota_media_tmdb"),
        col("qtd_votos_tmdb").cast("int").alias("qtd_votos_tmdb"),
        col("nota_media_imdb").cast("double").alias("nota_media_imdb"),
        col("qtd_votos_imdb").cast("int").alias("qtd_votos_imdb"),
    )
)

# Checagem de grão 
# Aqui a fato precisa ter exatamente uma linha por filme.
# comparo a contagem com dim_movies pra confirmar que os dois LEFT JOINs realmente preservaram o grão.
qtd_dim_movies = gold_dim_movies_ref.count()
qtd_fact = gold_fact_movies_performance.count()
print(f"Linhas em dim_movies: {qtd_dim_movies} | Linhas na fact_movies_performance: {qtd_fact} | Grão preservado: {qtd_dim_movies == qtd_fact}")

dq_check_unique("fact_movies_performance", "sk_movie_id_unico", gold_fact_movies_performance, ["sk_movie_id"])

gold_fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.fact_movies_performance")
display(spark.table(f"{gold_schema}.fact_movies_performance").limit(10))

## 10. gold.gold_genai_movies_context (Entrega 2 — tabela de contexto pro time de IA)

**Origem:** `gold.dim_movies` + `gold.fact_movies_performance` + `gold.dim_people` (via `gold.bridge_movie_person`)

**Objetivo:** Gerar o documento de contexto por filme que alimenta o Vector Search do assistente de IA (RAG) do time de Inteligência Artificial.

**Colunas:** `movie_id` (chave natural, = `id_filme`), `title`, `llm_context_document` (STRING, frase corrida seguindo o template do enunciado).

**Regras aplicadas:**
- **Origem de cada parte do template:** título/ano/sinopse vêm de `dim_movies`; receita/orçamento vêm de `fact_movies_performance`; atores principais e diretor vêm de `dim_people`, filtrando por `tipo_pessoa`, cruzado via `bridge_movie_person`. `dim_companies` não entra: o bridge de pessoa só conecta filme a pessoa física, e o template não pede produtora.
- **Atores principais:** como um filme pode ter dezenas de atores creditados, limito a lista aos 5 primeiros nomes em ordem alfabética — a origem não tem uma coluna de "ordem de bilheteria" pra eu saber quem são os atores mais importantes de fato, então a ordenação alfabética é só pra garantir um resultado reprodutível, não uma hierarquia real de importância. Decisão de projeto pra manter o documento gerado curto e útil pro RAG.
- **Diretor:** normalmente um só, mas junto todos os créditos de "Diretor" do filme (caso de co-direção), também ordenados alfabeticamente.
- **A "casca de banana" dos nulos (aviso explícito do enunciado):** `concat()` devolve NULL pra string inteira se qualquer campo envolvido for nulo. Antes de montar a frase final, aplico `coalesce()` (ou um `when` equivalente pros valores financeiros, que já ganham formatação de moeda) em CADA campo que pode legitimamente vir nulo — título, ano, receita, orçamento, atores, diretor, sinopse — cada um com um texto de fallback que faz sentido pro campo específico (ex.: "valor de receita não divulgado", "elenco não informado"). Só depois disso a concatenação final é segura.
- Receita e orçamento NULL são a norma, não exceção: só ~3,3% dos filmes têm receita e ~8,3% têm orçamento preenchidos (medido na Silver). Por isso o fallback textual desses dois campos é especialmente importante — sem ele, a maioria dos filmes desapareceria silenciosamente da tabela de contexto.
- `dq_check` dedicado confirma que `llm_context_document` nunca fica nulo em nenhuma linha (validação direta de que a proteção contra a casca de banana funcionou), além do `dq_check_unique` em `movie_id`.

In [0]:
from pyspark.sql.functions import (
    col, concat, concat_ws, lit, coalesce, collect_list, array_sort, slice as spark_slice, when, format_number
)

#Base: dim_movies e fact_movies_performance (título, ano, sinopse, receita, orçamento)
tb_dim_movies = spark.table(f"{gold_schema}.dim_movies").select(
    "sk_movie_id", "id_filme", "titulo", "ano_lancamento", "sinopse"
)
tb_fact = spark.table(f"{gold_schema}.fact_movies_performance").select(
    "sk_movie_id", "receita_usd", "orcamento_usd"
)

# Atores principais e diretor, via bridge_movie_person e dim_people
tb_bridge_movie_person = spark.table(f"{gold_schema}.bridge_movie_person")
tb_dim_people = spark.table(f"{gold_schema}.dim_people")

pessoas_por_filme = (
    tb_bridge_movie_person
    .join(tb_dim_people, on="sk_person_id", how="inner")
    .select("sk_movie_id", "nome_pessoa", "tipo_pessoa")
)

# Para definir o elenco, pensei que como um filme pode ter dezenas de atores creditados, limitei aos 5 primeiros em ordem
# alfabética (a origem não tem uma coluna de "ordem de bilheteria" pra eu saber quem são os atores mais importantes de verdade).
# O texto gerado assim fica curto e útil pro RAG, em vez de uma lista enorme com o elenco inteiro.
atores_agregados = (
    pessoas_por_filme
    .filter(col("tipo_pessoa") == "Ator")
    .groupBy("sk_movie_id")
    .agg(array_sort(collect_list("nome_pessoa")).alias("_atores_ordenados"))
    .withColumn("atores_principais", concat_ws(", ", spark_slice(col("_atores_ordenados"), 1, 5)))
    .select("sk_movie_id", "atores_principais")
)

# Como o diretor geralmente é só um, mas alguns filmes têm co-direção, eu decidi juntar todos os créditos de Diretor dos filme, também ordenados #alfabeticamente.
diretor_agregado = (
    pessoas_por_filme
    .filter(col("tipo_pessoa") == "Diretor")
    .groupBy("sk_movie_id")
    .agg(concat_ws(", ", array_sort(collect_list("nome_pessoa"))).alias("diretor"))
)

#Para montar a base completa eu realziei um LEFT JOIN em tudo já que nenhum desses dados é garantido pra todo filme.
tb_contexto_base = (
    tb_dim_movies
    .join(tb_fact, on="sk_movie_id", how="left")
    .join(atores_agregados, on="sk_movie_id", how="left")
    .join(diretor_agregado, on="sk_movie_id", how="left")
)

# Aqui tomei cuidado em relação a "casca de banana" que foi sinalizado no material.
# Pesquisei e vi que concat()/concat_ws() devolvem NULL pra STRING INTEIRA se qualquer campo envolvido for nulo. Por isso, antes de montar a
# frase final, eu decidi aplicar coalesce/"when" em cada campo que pode vir nulo, com um texto de fallback que faça sentido pro campo
# Apenas depois disso considerei seguro concatenar sem risco de perder o filme inteiro da tabela de contexto silenciosamente.
titulo_seguro = coalesce(col("titulo"), lit("Título não informado"))
ano_seguro = coalesce(col("ano_lancamento").cast("string"), lit("ano não informado"))

# Agora sobre a receita, depois de calcular na silver, percebi que a a maioria dos filmes não tem esses valores preenchidos 
# Decidi aplicar o format_number comoseparador de milhar para ajudando a legibilidade do texto pro RAG.
receita_segura = when(
    col("receita_usd").isNotNull(), concat(lit("US$ "), format_number(col("receita_usd"), 2))
).otherwise(lit("valor de receita não divulgado"))

orcamento_seguro = when(
    col("orcamento_usd").isNotNull(), concat(lit("US$ "), format_number(col("orcamento_usd"), 2))
).otherwise(lit("valor de orçamento não divulgado"))

atores_seguro = coalesce(col("atores_principais"), lit("elenco não informado"))
diretor_seguro = coalesce(col("diretor"), lit("diretor não informado"))
sinopse_segura = coalesce(col("sinopse"), lit("sinopse não disponível"))

#Concatenação final
gold_genai_movies_context = (
    tb_contexto_base
    .withColumn(
        "llm_context_document",
        concat(
            lit("O filme "), titulo_seguro,
            lit(", lançado no ano de "), ano_seguro,
            lit(", faturou "), receita_segura,
            lit(" e teve um custo de "), orcamento_seguro,
            lit(". Estrelado por "), atores_seguro,
            lit(" e dirigido por "), diretor_seguro,
            lit(", o filme possui a seguinte sinopse: "), sinopse_segura,
            lit("."),
        )
    )
    .select(
        col("id_filme").alias("movie_id"),
        titulo_seguro.alias("title"),
        "llm_context_document",
    )
)

# Checagem de qualidade
dq_check(
    "gold_genai_movies_context", "llm_context_document nunca nulo",
    gold_genai_movies_context, col("llm_context_document").isNotNull()
)
dq_check_unique("gold_genai_movies_context", "movie_id_unico", gold_genai_movies_context, ["movie_id"])

gold_genai_movies_context.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{gold_schema}.gold_genai_movies_context")
display(spark.table(f"{gold_schema}.gold_genai_movies_context").limit(5))

## Desafio de Analytics 

**Perguntas de Negócio:** 
- 1. Qual é a receita total (em R$) somada de todos os filmes da base? 
- 2. Quais são os 5 filmes com maior popularidade? Mostre título e valor de popularidade. 
- 3. Quantos filmes cada gênero possui? Liste do maior para o menor volume. 
- 4. Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição de cada um no 
ranking (RANK()). 
- 5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos*? 
- 6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos*? 

-*Para o recorte dos últimos 2 e 5 anos, considere como data limite superior a data de lançamento realizada 
mais recente na base (ignorando registros com datas futuras ou não lançadas).  

In [0]:
#1. Qual é a receita total (em R$) somada de todos os filmes da base?

from pyspark.sql.functions import sum as spark_sum, format_number, concat, lit

tb_fact = spark.table(f"{gold_schema}.fact_movies_performance")

receita_total_brl = tb_fact.agg(
    spark_sum("receita_brl").alias("receita_total_brl")
)

receita_total_formatada = receita_total_brl.select(
    concat(lit("R$ "), format_number("receita_total_brl", 2)).alias("receita_total_formatada")
)

display(receita_total_brl)
display(receita_total_formatada)

In [0]:
#2. Quais são os 5 filmes com maior popularidade? Mostre título e valor de popularidade.

from pyspark.sql.functions import col

tb_dim_movies = spark.table(f"{gold_schema}.dim_movies")
tb_fact = spark.table(f"{gold_schema}.fact_movies_performance")

top5_popularidade = (
    tb_dim_movies
    .join(tb_fact, on="sk_movie_id", how="inner")
    .filter(col("popularidade").isNotNull())
    .select("titulo", "popularidade")
    .orderBy(col("popularidade").desc())
    .limit(5)
)

display(top5_popularidade)

In [0]:
#3. Quantos filmes cada gênero possui? Liste do maior para o menor volume.

from pyspark.sql.functions import col, count

tb_bridge_genre = spark.table(f"{gold_schema}.bridge_movie_genre")
tb_dim_genres = spark.table(f"{gold_schema}.dim_genres")

filmes_por_genero = (
    tb_bridge_genre
    .join(tb_dim_genres, on="sk_genre_id", how="inner")
    .groupBy("nome_genero")
    .agg(count("sk_movie_id").alias("qtd_filmes"))
    .orderBy(col("qtd_filmes").desc())
)

display(filmes_por_genero)

In [0]:
#4. Para os 10 filmes de maior receita, mostre título, receita (em US$ e R$) e a posição de cada um no ranking (RANK()).

from pyspark.sql import Window
from pyspark.sql.functions import col, rank

tb_dim_movies = spark.table(f"{gold_schema}.dim_movies")
tb_fact = spark.table(f"{gold_schema}.fact_movies_performance")

janela_rank_receita = Window.orderBy(col("receita_usd").desc())

top10_receita = (
    tb_dim_movies
    .join(tb_fact, on="sk_movie_id", how="inner")
    .filter(col("receita_usd").isNotNull())
    .withColumn("posicao_ranking", rank().over(janela_rank_receita))
    .select("titulo", "receita_usd", "receita_brl", "posicao_ranking")
    .orderBy(col("posicao_ranking"))
    .limit(10)
)

display(top10_receita)

In [0]:
#5. Qual ator teve a maior quantidade de participações nos filmes lançados nos últimos 2 anos*?

from pyspark.sql.functions import col, count, add_months, current_date, lit

tb_dim_movies = spark.table(f"{gold_schema}.dim_movies")

data_referencia = (
    tb_dim_movies
    .filter((col("status_filme") == "Lançado") & (col("data_lancamento") <= current_date()))
    .agg({"data_lancamento": "max"})
    .collect()[0][0]
)
print(f"Data de referência (lançamento mais recente realizado na base): {data_referencia}")

filmes_ultimos_2_anos = (
    tb_dim_movies
    .filter(
        (col("status_filme") == "Lançado")
        & (col("data_lancamento") <= lit(data_referencia))
        & (col("data_lancamento") >= add_months(lit(data_referencia), -24))
    )
    .select("sk_movie_id")
)

tb_bridge_person = spark.table(f"{gold_schema}.bridge_movie_person")
tb_dim_people = spark.table(f"{gold_schema}.dim_people")

atores_participacoes = (
    filmes_ultimos_2_anos
    .join(tb_bridge_person, on="sk_movie_id", how="inner")
    .join(tb_dim_people.filter(col("tipo_pessoa") == "Ator"), on="sk_person_id", how="inner")
    .groupBy("nome_pessoa")
    .agg(count("sk_movie_id").alias("qtd_participacoes"))
    .orderBy(col("qtd_participacoes").desc())
)

display(atores_participacoes.limit(10))

In [0]:
#6. Qual a produtora de filmes teve o maior Lucro nos últimos 5 anos*?

from pyspark.sql.functions import col, sum as spark_sum, add_months, lit

tb_dim_movies = spark.table(f"{gold_schema}.dim_movies")

filmes_ultimos_5_anos = (
    tb_dim_movies
    .filter(
        (col("status_filme") == "Lançado")
        & (col("data_lancamento") <= lit(data_referencia))
        & (col("data_lancamento") >= add_months(lit(data_referencia), -60))
    )
    .select("sk_movie_id")
)

tb_bridge_company = spark.table(f"{gold_schema}.bridge_movie_company")
tb_dim_companies = spark.table(f"{gold_schema}.dim_companies")
tb_fact = spark.table(f"{gold_schema}.fact_movies_performance").select("sk_movie_id", "lucro_usd")

lucro_por_produtora = (
    filmes_ultimos_5_anos
    .join(tb_bridge_company, on="sk_movie_id", how="inner")
    .join(tb_dim_companies, on="sk_company_id", how="inner")
    .join(tb_fact, on="sk_movie_id", how="inner")
    .filter(col("lucro_usd").isNotNull())
    .groupBy("nome_produtora")
    .agg(spark_sum("lucro_usd").alias("lucro_total_usd"))
    .orderBy(col("lucro_total_usd").desc())
)

display(lucro_por_produtora.limit(10))